# Фото проводник – демонстрация библиотеки PhotoExplorer.Core

В этом ноутбуке показано, как работает основная логика программы без графического интерфейса:
обзор папок, сортировка, поиск, чтение EXIF, миниатюры и их кеш в SQLite, а также консольная утилита `photos`.

**Перед запуском** соберите решение из корня репозитория:

```
dotnet build
```

## 1. Подключение библиотеки

Сначала подключаем пакеты, от которых зависит `PhotoExplorer.Core`, а потом саму собранную библиотеку.

In [ ]:
#r "nuget: SkiaSharp, 3.119.0"
#r "nuget: MetadataExtractor, 2.9.3"
#r "nuget: Microsoft.Data.Sqlite, 10.0.12"
#r "../src/PhotoExplorer.Core/bin/Debug/net10.0/PhotoExplorer.Core.dll"

In [ ]:
using System.Diagnostics;
using System.IO;
using PhotoExplorer.Core;
using PhotoExplorer.Core.Models;
using SkiaSharp;

## 2. Тестовые изображения

Чтобы ноутбук не зависел от чужих фотографий, создаём папку `sample` с картинками прямо здесь, через SkiaSharp:
разные размеры, форматы (JPEG, PNG, WebP) и вложенные папки.

In [ ]:
var sampleDir = Path.GetFullPath("sample");
if (Directory.Exists(sampleDir))
    Directory.Delete(sampleDir, true);

// Рисует простую картинку: градиент и круг
void CreateImage(string relativePath, int width, int height, SKColor color, SKEncodedImageFormat format)
{
    var path = Path.Combine(sampleDir, relativePath);
    Directory.CreateDirectory(Path.GetDirectoryName(path));

    using var bitmap = new SKBitmap(width, height);
    using (var canvas = new SKCanvas(bitmap))
    {
        using var paint = new SKPaint
        {
            Shader = SKShader.CreateLinearGradient(new SKPoint(0, 0), new SKPoint(0, height),
                new[] { color, SKColors.White }, SKShaderTileMode.Clamp)
        };
        canvas.DrawRect(0, 0, width, height, paint);
        using var circle = new SKPaint { Color = SKColors.OrangeRed, IsAntialias = true };
        canvas.DrawCircle(width / 2f, height / 2f, Math.Min(width, height) / 4f, circle);
    }
    using var data = bitmap.Encode(format, 90);
    File.WriteAllBytes(path, data.ToArray());
}

CreateImage("img1.jpg", 1200, 800, SKColors.SteelBlue, SKEncodedImageFormat.Jpeg);
CreateImage("img2.jpg", 800, 1200, SKColors.SeaGreen, SKEncodedImageFormat.Jpeg);
CreateImage("img10.jpg", 1600, 900, SKColors.Purple, SKEncodedImageFormat.Jpeg);
CreateImage("схема.png", 640, 480, SKColors.Goldenrod, SKEncodedImageFormat.Png);
CreateImage("стикер.webp", 512, 512, SKColors.HotPink, SKEncodedImageFormat.Webp);
CreateImage("Отпуск/отпуск_море_1.jpg", 2000, 1333, SKColors.DeepSkyBlue, SKEncodedImageFormat.Jpeg);
CreateImage("Отпуск/отпуск_море_2.jpg", 1333, 2000, SKColors.Teal, SKEncodedImageFormat.Jpeg);
CreateImage("Отпуск/Горы/отпуск_горы.png", 1000, 700, SKColors.SlateGray, SKEncodedImageFormat.Png);
File.WriteAllText(Path.Combine(sampleDir, "заметки.txt"), "это не картинка");

Directory.GetFiles(sampleDir, "*", SearchOption.AllDirectories).Select(f => Path.GetRelativePath(sampleDir, f))

## 3. Список файлов и сортировка

`FolderBrowser` возвращает только поддерживаемые изображения (файл `заметки.txt` не попадёт в список).
`ImageSorter` умеет естественную сортировку по имени: `img2` идёт раньше `img10`.

In [ ]:
var settings = new AppSettings();
var browser = new FolderBrowser(settings);
var files = browser.GetImages(sampleDir);

ImageSorter.Sort(files, SortField.Name).Select(f => f.Name)

In [ ]:
// По размеру файла, от большего к меньшему
ImageSorter.Sort(files, SortField.Size, descending: true)
    .Select(f => new { f.Name, Размер = Formatting.FormatSize(f.Size), f.Extension })

In [ ]:
// Подпапки для дерева папок
browser.GetSubfolders(sampleDir)

## 4. Поиск

Поиск работает по подстроке или по маске (`*` и `?`), без учёта регистра. Результаты приходят порциями
через `IAsyncEnumerable`, поэтому программа может показывать их по мере нахождения.

In [ ]:
var search = new ImageSearch(browser);

async Task<List<string>> Find(SearchOptions options)
{
    var result = new List<string>();
    await foreach (var file in search.SearchAsync(sampleDir, options))
        result.Add(Path.GetRelativePath(sampleDir, file.FullPath));
    return result;
}

display(await Find(new SearchOptions { Name = "img?.jpg" }));                       // маска
display(await Find(new SearchOptions { Name = "*отпуск*" }));                       // в текущей папке ничего
display(await Find(new SearchOptions { Name = "*отпуск*", Recursive = true }));     // рекурсивно
display(await Find(new SearchOptions { Recursive = true, Extensions = ["png"] }));  // фильтр по типу

## 5. Метаданные

`MetadataReader` читает разрешение, формат и EXIF. У сгенерированных картинок EXIF нет,
поэтому заполнены только базовые поля.

In [ ]:
MetadataReader.Read(Path.Combine(sampleDir, "img10.jpg"))

## 6. Миниатюры и кеш

Строим миниатюры через `ThumbnailCache` (база SQLite создаётся рядом, в `sample/thumbs.db`)
и выводим их прямо в ноутбук как HTML с картинками в base64.

In [ ]:
var cache = new ThumbnailCache(Path.Combine(sampleDir, "thumbs.db"));
var allFiles = await Find(new SearchOptions { Recursive = true });

// Строит миниатюры всех файлов и возвращает затраченное время
async Task<TimeSpan> BuildThumbnails(bool show)
{
    var html = new System.Text.StringBuilder();
    var watch = Stopwatch.StartNew();
    foreach (var relative in allFiles)
    {
        var thumb = await cache.GetOrCreateAsync(Path.Combine(sampleDir, relative), 160);
        var base64 = Convert.ToBase64String(thumb.Data);
        html.Append($"<div style='display:inline-block;margin:4px;text-align:center;font-size:12px'>" +
                    $"<img src='data:image/jpeg;base64,{base64}'/><br/>{Path.GetFileName(relative)}<br/>" +
                    $"{thumb.OriginalWidth} x {thumb.OriginalHeight}</div>");
    }
    watch.Stop();
    if (show)
        html.ToString().DisplayAs("text/html");
    return watch.Elapsed;
}

var firstTime = await BuildThumbnails(show: true);
$"Первый проход: {firstTime.TotalMilliseconds:0} мс, построено миниатюр: {cache.Misses}"

Повторный запрос тех же миниатюр – теперь они берутся из базы, а не строятся заново:

In [ ]:
var secondTime = await BuildThumbnails(show: false);
var stats = cache.GetStats();
$"Второй проход: {secondTime.TotalMilliseconds:0} мс. Записей в кеше: {stats.Count}, " +
$"попаданий: {stats.Hits}, промахов: {stats.Misses}, процент попаданий: {stats.HitRate:0}%"

## 7. Изменённый файл получает новую миниатюру

Кеш хранит размер и дату изменения файла. Перезапишем `img1.jpg` другой картинкой – запись станет устаревшей,
и миниатюра построится заново.

In [ ]:
var img1 = Path.Combine(sampleDir, "img1.jpg");
var before = await cache.GetOrCreateAsync(img1, 160);

CreateImage("img1.jpg", 600, 900, SKColors.Crimson, SKEncodedImageFormat.Jpeg);
File.SetLastWriteTime(img1, DateTime.Now.AddMinutes(1));

int missesBefore = cache.Misses;
var after = await cache.GetOrCreateAsync(img1, 160);

($"<img src='data:image/jpeg;base64,{Convert.ToBase64String(before.Data)}'/> &rarr; " +
 $"<img src='data:image/jpeg;base64,{Convert.ToBase64String(after.Data)}'/>").DisplayAs("text/html");
$"Было {before.OriginalWidth} x {before.OriginalHeight}, стало {after.OriginalWidth} x {after.OriginalHeight}. " +
$"Миниатюра построена заново: {cache.Misses > missesBefore}"

Удалённые файлы вычищаются из кеша методом `RemoveMissing`:

In [ ]:
File.Delete(Path.Combine(sampleDir, "стикер.webp"));
var removed = cache.RemoveMissing();
$"Удалено записей: {removed}, осталось: {cache.GetStats().Count}"

## 8. Консольная утилита

То же самое доступно из командной строки. Запускаем собранную `photos.dll` через `dotnet`
и используем отдельную базу кеша (`--cache`), чтобы не трогать основную.

In [ ]:
var cliPath = Path.GetFullPath("../src/PhotoExplorer.Cli/bin/Debug/net10.0/photos.dll");
var cliCache = Path.Combine("sample", "cli-thumbs.db");

// Запускает CLI и возвращает то, что он напечатал
string Photos(params string[] args)
{
    var info = new ProcessStartInfo("dotnet") { RedirectStandardOutput = true, RedirectStandardError = true };
    // Папка ноутбука – рабочая, поэтому пути можно писать относительно неё
    info.WorkingDirectory = Path.GetDirectoryName(sampleDir);
    info.StandardOutputEncoding = System.Text.Encoding.UTF8;
    info.ArgumentList.Add(cliPath);
    foreach (var arg in args)
        info.ArgumentList.Add(arg);
    using var process = Process.Start(info);
    var output = process.StandardOutput.ReadToEnd() + process.StandardError.ReadToEnd();
    process.WaitForExit();
    return $"> photos {string.Join(" ", args)}   (код выхода {process.ExitCode})\n{output}";
}

Console.WriteLine(Photos("list", "sample", "--sort", "size", "--desc"));

In [ ]:
Console.WriteLine(Photos("search", "sample", "--name", "*отпуск*", "--recursive"));

In [ ]:
Console.WriteLine(Photos("info", Path.Combine("sample", "img10.jpg")));

In [ ]:
Console.WriteLine(Photos("thumbs", "build", "sample", "--recursive", "--size", "160", "--cache", cliCache));
Console.WriteLine(Photos("thumbs", "build", "sample", "--recursive", "--size", "160", "--cache", cliCache));

In [ ]:
Console.WriteLine(Photos("cache", "stats", "--cache", cliCache));
Console.WriteLine(Photos("list", Path.Combine("sample", "нет такой папки")));